In [ ]:
from time import monotonic
import time, json
from pathlib import Path

import requests
import pandas as pd

BASE_URL = "https://oqmd.org/oqmdapi/formationenergy"

PAGE_SIZE = 250 #Increase or lower this depending on your needs and the stability of your connection. 
# smaller may be more relu=iable, 100 was the most stable for me.
CHECKPOINT_CSV = Path("oqmd_formation_energies_checkpoint.csv")
CHECKPOINT_JSON = Path("oqmd_formation_energies_checkpoint.json")
OUTPUT_CSV = Path("oqmd_formation_energies_clean.csv")

FIELDS = [
    "name",
    "entry_id",
    "formationenergy_id",
    "duplicate_entry_id",
    "icsd_id",
    "prototype",
    "spacegroup",
    "composition_generic",
    "ntypes",
    "natoms",
    "volume",
    "unit_cell",
    "sites",
    "delta_e",      # formation energy, eV/atom
    "stability",    # energy above hull, eV/atom
    "band_gap",
    "fit",
    "calculation_label",
]

# Edit this for narrower searches, e.g.:
# FILTER = "stability<0.1 AND ntypes>=2"
# FILTER = "element_set=(Li-Na),O AND stability<0.2"
FILTER = None


PARAMS_BASE = {
    "fields": ",".join(FIELDS),
    "limit": PAGE_SIZE,
    "format": "json",
    "noduplicate": "False",
}


def get_with_retries(url, params, max_retries=8):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=(10, 300))
            if r.status_code == 200:
                return r
            print(f"Attempt {attempt + 1}: HTTP {r.status_code}")
            print(r.text[:500])
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: {type(e).__name__} - {e}")

        sleep_time = 5 * (attempt + 1)
        print(f"Retrying in {sleep_time} seconds...")
        time.sleep(sleep_time)

    raise RuntimeError("Request failed after all retry attempts")


def normalise_record(row):
    return {
        "name": row.get("name"),
        "entry_id": row.get("entry_id"),
        "formationenergy_id": row.get("formationenergy_id"),
        "duplicate_entry_id": row.get("duplicate_entry_id"),
        "icsd_id": row.get("icsd_id"),
        "prototype": row.get("prototype"),
        "spacegroup": row.get("spacegroup"),
        "composition_generic": row.get("composition_generic"),
        "ntypes": row.get("ntypes"),
        "natoms": row.get("natoms"),
        "volume_A3": row.get("volume"),
        "formation_energy_eV_per_atom": row.get("delta_e"),
        "stability_eV_per_atom": row.get("stability"),
        "band_gap_eV": row.get("band_gap"),
        "fit": row.get("fit"),
        "calculation_label": row.get("calculation_label"),
        # Keep structure-rich nested fields as JSON strings for CSV safety
        "unit_cell_json": json.dumps(row.get("unit_cell"), ensure_ascii=False),
        "sites_json": json.dumps(row.get("sites"), ensure_ascii=False),
    }


if CHECKPOINT_CSV.exists():
    df_checkpoint = pd.read_csv(CHECKPOINT_CSV)
    records = df_checkpoint.to_dict("records")
    print(f"Loaded checkpoint with {len(records)} records.")
else:
    records = []

if CHECKPOINT_JSON.exists():
    checkpoint = json.loads(CHECKPOINT_JSON.read_text())
    offset = checkpoint.get("offset", 0)
    page_number = checkpoint.get("page_number", 0)
    total_entries = checkpoint.get("total_entries")
    print(f"Resuming from page {page_number}, offset {offset}.")
else:
    offset = 0
    page_number = 0
    total_entries = None

start = monotonic()

while True:
    params = dict(PARAMS_BASE)
    params["offset"] = offset
    if FILTER:
        params["filter"] = FILTER

    response = get_with_retries(BASE_URL, params)

    try:
        payload = response.json()
    except ValueError:
        print("Failed to decode JSON.")
        print(response.text[:1000])
        break

    if payload.get("response_message") not in (None, "OK"):
        print("Unexpected response:", payload.get("response_message"))

    meta = payload.get("meta", {})
    data = payload.get("data", [])

    if total_entries is None:
        total_entries = meta.get("data_available")
        print(f"Total entries available: {total_entries}")

    if not data:
        print("No more data returned.")
        break

    page_number += 1
    new_records = [normalise_record(row) for row in data]
    records.extend(new_records)

    df = pd.DataFrame(records)
    df.to_csv(CHECKPOINT_CSV, index=False)

    offset += len(data)

    CHECKPOINT_JSON.write_text(json.dumps({
        "offset": offset,
        "page_number": page_number,
        "total_entries": total_entries,
        "records_downloaded": len(records),
        "filter": FILTER,
        "fields": FIELDS,
    }, indent=2))

    print(
        f"Page {page_number} complete | "
        f"Downloaded: {len(records)} / {total_entries}"
    )

    if not meta.get("more_data_available", False):
        break

end = monotonic()

df = pd.DataFrame(records)

numeric_cols = [
    "formation_energy_eV_per_atom",
    "stability_eV_per_atom",
    "band_gap_eV",
    "volume_A3",
    "natoms",
    "ntypes",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["formation_energy_eV_per_atom"])
df = df.drop_duplicates(subset=["entry_id", "formationenergy_id"])
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Query took {end - start:.2f} seconds")
print(f"Final cleaned records: {len(df)}")
print(df.head())